# E46 — TRT int8 on the T4 (Colab): build → run → save engine

Colab's only job: take the **pre-quantized int8 model** (made offline), **build the TRT int8 engine on the T4**, **run it** (F1 + timing), and **save the engine** to ship.

**Setup:** Runtime ▸ Change runtime type ▸ **T4 GPU**. Put in Google Drive `MyDrive/dacon_trt/`:
- `model_int8_pruned.onnx` — the pre-quantized int8 Q/DQ model (provided)
- `val_tokens_pruned.npz` — 3.5k labelled val slice

In [ ]:
# 1. install (pin TRT 10.7 = version we ship; torch preinstalled on Colab)
import time; t0=time.time()
!pip install -q "tensorrt==10.7.0" numpy scikit-learn
import tensorrt as trt; print('TensorRT', trt.__version__)
print(f'install {time.time()-t0:.0f}s')

In [ ]:
# 2. confirm T4 + mount Drive
import subprocess
gpu = subprocess.run(['nvidia-smi','--query-gpu=name,compute_cap','--format=csv,noheader'],capture_output=True,text=True).stdout
print(gpu); assert 'T4' in gpu, 'Set runtime to T4 GPU'
from google.colab import drive; drive.mount('/content/drive')
SRC='/content/drive/MyDrive/dacon_trt'

In [ ]:
# 3. helpers + load val (torch = preinstalled, used only for GPU memory)
import numpy as np, torch, tensorrt as trt, time
from sklearn.metrics import f1_score
SEQ, NCLS, B = 512, 14, 64
logger = trt.Logger(trt.Logger.WARNING)
d = np.load(f'{SRC}/val_tokens_pruned.npz')
val_ids, val_mask, val_y = d['val_ids'], d['val_mask'], d['val_y']
_FLAGS = (1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
          if hasattr(trt.NetworkDefinitionCreationFlag,'EXPLICIT_BATCH') else 0)

def build_int8(onnx_path):
    b=trt.Builder(logger); net=b.create_network(_FLAGS); p=trt.OnnxParser(net,logger)
    if not p.parse(open(onnx_path,'rb').read()):
        for i in range(p.num_errors): print(p.get_error(i))
        raise RuntimeError('parse failed')
    cfg=b.create_builder_config(); cfg.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE,4<<30)
    cfg.set_flag(trt.BuilderFlag.FP16); cfg.set_flag(trt.BuilderFlag.INT8)
    prof=b.create_optimization_profile()
    for n in ('input_ids','attention_mask'): prof.set_shape(n,(1,SEQ),(B,SEQ),(B,SEQ))
    cfg.add_optimization_profile(prof)
    t0=time.time(); blob=b.build_serialized_network(net,cfg); bs=time.time()-t0
    assert blob is not None,'engine build failed'
    return trt.Runtime(logger).deserialize_cuda_engine(blob), blob, bs

def infer(engine, ids, mask):
    ctx=engine.create_execution_context()
    di=torch.zeros(B,SEQ,dtype=torch.int64,device='cuda'); dm=torch.zeros(B,SEQ,dtype=torch.int64,device='cuda')
    do=torch.zeros(B,NCLS,dtype=torch.float32,device='cuda'); logits=np.empty((len(ids),NCLS),np.float32)
    torch.cuda.synchronize(); t0=time.time()
    for s in range(0,len(ids),B):
        e=min(s+B,len(ids)); b=e-s
        di[:b]=torch.from_numpy(ids[s:e].astype(np.int64)).cuda(); dm[:b]=torch.from_numpy(mask[s:e].astype(np.int64)).cuda()
        ctx.set_input_shape('input_ids',(b,SEQ)); ctx.set_input_shape('attention_mask',(b,SEQ))
        ctx.set_tensor_address('input_ids',di.data_ptr()); ctx.set_tensor_address('attention_mask',dm.data_ptr()); ctx.set_tensor_address('logits',do.data_ptr())
        ctx.execute_async_v3(torch.cuda.current_stream().cuda_stream); torch.cuda.synchronize()
        logits[s:e]=do[:b].cpu().numpy()
    return logits, time.time()-t0

In [ ]:
# 4. build int8 engine on the T4 -> run -> save engine
eng, blob, build_s = build_int8(f'{SRC}/model_int8_pruned.onnx')
lo, inf_s = infer(eng, val_ids, val_mask)
f1 = f1_score(val_y, lo.argmax(-1), average='macro'); per = inf_s/len(val_ids)
print(f'int8: build {build_s:.1f}s | infer {inf_s:.1f}s/{len(val_ids)} | F1 {f1:.4f} | classes {len(np.unique(lo.argmax(-1)))}/14')
print(f'proj 30k on T4 = build {build_s:.0f}s + infer {per*30000:.0f}s = {build_s+per*30000:.0f}s (cap 600s) -> {"FITS" if build_s+per*30000<600 else "OVER"}')
# save the built T4 engine (ship candidate) back to Drive
open(f'{SRC}/model_int8p_t4.engine','wb').write(blob)
print(f'saved engine {blob.nbytes/1e6:.0f}MB -> {SRC}/model_int8p_t4.engine')